In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

CHANGES_DIR = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\analysis_results\time_changes"
)

def load_changes(filename):
    frame = pd.read_csv(
        CHANGES_DIR / filename,
        keep_default_na=False,
        na_values=[""],
        dtype={
            "iso3": "string",
            "country": "string",
            "year_start": "Int64",
            "year_end": "Int64",
            "either_endpoint_flagged": "boolean",
        },
    )

    required = [
        "iso3",
        "year_start",
        "year_end",
        "delta_female_hours",
        "delta_female_lfpr_pp",
        "either_endpoint_flagged",
    ]

    if frame[required].isna().any().any():
        raise ValueError(f"Missing required values: {filename}")

    if frame.duplicated(["iso3", "year_start", "year_end"]).any():
        raise ValueError(f"Duplicate observation pairs: {filename}")

    if frame["year_end"].le(frame["year_start"]).any():
        raise ValueError(f"Invalid observation intervals: {filename}")

    frame["freed_hours_per_day"] = -frame["delta_female_hours"]
    frame["freed_minutes_per_day"] = frame["freed_hours_per_day"] * 60
    frame["female_lfpr_change_pp"] = frame["delta_female_lfpr_pp"]

    return frame


first_last = load_changes("first_last_changes.csv")
adjacent = load_changes("adjacent_pair_changes.csv")

if first_last["iso3"].duplicated().any():
    raise ValueError("Expected one first-to-last pair per entity.")

opportunity_views = {
    "First-to-last: all": first_last,
    "First-to-last: exclude conflicts": first_last.loc[
        ~first_last["either_endpoint_flagged"]
    ].copy(),
    "Adjacent: exclude conflicts": adjacent.loc[
        ~adjacent["either_endpoint_flagged"]
    ].copy(),
}

display(pd.DataFrame([
    {
        "view": name,
        "pairs": len(frame),
        "entities": frame["iso3"].nunique(),
    }
    for name, frame in opportunity_views.items()
]))

,view,pairs,entities
0,First-to-last: all,45,45
1,First-to-last: exclude conflicts,41,41
2,Adjacent: exclude conflicts,83,42


In [9]:
CLASSIFICATION_RULES = {
    "Numerical direction": {
        "time_hours": 1e-6,
        "lfpr_pp": 1e-6,
    },
    "10 minutes and 1 percentage point": {
        "time_hours": 10 / 60,
        "lfpr_pp": 1.0,
    },
}

def classify_changes(frame, time_threshold, lfpr_threshold):
    classified = frame.copy()

    classified["time_status"] = "Approximately unchanged"
    classified.loc[
        classified["freed_hours_per_day"].gt(time_threshold),
        "time_status",
    ] = "Female unpaid time decreased"
    classified.loc[
        classified["freed_hours_per_day"].lt(-time_threshold),
        "time_status",
    ] = "Female unpaid time increased"

    classified["participation_status"] = "Approximately unchanged"
    classified.loc[
        classified["female_lfpr_change_pp"].gt(lfpr_threshold),
        "participation_status",
    ] = "Participation increased"
    classified.loc[
        classified["female_lfpr_change_pp"].lt(-lfpr_threshold),
        "participation_status",
    ] = "Participation decreased"

    return classified


classified_views = {}
freed_time_summary_rows = []

for view_name, frame in opportunity_views.items():
    for rule_name, thresholds in CLASSIFICATION_RULES.items():
        classified = classify_changes(
            frame,
            thresholds["time_hours"],
            thresholds["lfpr_pp"],
        )

        classified_views[(view_name, rule_name)] = classified

        freed = classified.loc[
            classified["time_status"].eq("Female unpaid time decreased")
        ]

        counts = freed["participation_status"].value_counts()
        increased = int(counts.get("Participation increased", 0))
        unchanged = int(counts.get("Approximately unchanged", 0))
        decreased = int(counts.get("Participation decreased", 0))

        freed_time_summary_rows.append({
            "view": view_name,
            "rule": rule_name,
            "freed_time_pairs": len(freed),
            "entities": freed["iso3"].nunique(),
            "participation_increased": increased,
            "participation_approximately_unchanged": unchanged,
            "participation_decreased": decreased,
            "pct_without_increase_under_rule": (
                (unchanged + decreased) / len(freed) * 100
                if len(freed) else np.nan
            ),
        })

freed_time_summary = pd.DataFrame(freed_time_summary_rows)

with pd.option_context("display.max_columns", None):
    display(freed_time_summary.round(2))

for rule_name in CLASSIFICATION_RULES:
    main = classified_views[
        ("First-to-last: exclude conflicts", rule_name)
    ]

    print(f"\nJoint change directions: {rule_name}")
    display(pd.crosstab(
        main["time_status"],
        main["participation_status"],
        rownames=["Female unpaid time"],
        colnames=["Female labor force participation"],
    ))

,view,rule,freed_time_pairs,entities,participation_increased,participation_approximately_unchanged,participation_decreased,pct_without_increase_under_rule
0,First-to-last: all,Numerical direction,37,37,23,0,14,37.84
1,First-to-last: all,10 minutes and 1 percentage point,31,31,18,7,6,41.94
2,First-to-last: exclude conflicts,Numerical direction,33,33,23,0,10,30.30
3,First-to-last: exclude conflicts,10 minutes and 1 percentage point,27,27,18,5,4,33.33
4,Adjacent: exclude conflicts,Numerical direction,55,34,36,0,19,34.55
5,Adjacent: exclude conflicts,10 minutes and 1 percentage point,35,29,20,10,5,42.86



Joint change directions: Numerical direction


Female labor force participation,Participation decreased,Participation increased
Female unpaid time,,
Female unpaid time decreased,10,23
Female unpaid time increased,6,2



Joint change directions: 10 minutes and 1 percentage point


Female labor force participation,Approximately unchanged,Participation decreased,Participation increased
Female unpaid time,,,
Approximately unchanged,1,1,4
Female unpaid time decreased,5,4,18
Female unpaid time increased,1,5,2


In [10]:
main_cases = classified_views[
    (
        "First-to-last: exclude conflicts",
        "10 minutes and 1 percentage point",
    )
].copy()

freed_time_cases = main_cases.loc[
    main_cases["time_status"].eq("Female unpaid time decreased")
].copy()

CASE_COLUMNS = [
    "country",
    "year_start",
    "year_end",
    "interval_years",
    "female_unpaid_hours_start",
    "female_unpaid_hours_end",
    "freed_minutes_per_day",
    "female_lfpr_pct_start",
    "female_lfpr_pct_end",
    "female_lfpr_change_pp",
    "survey_comparability",
]

for status in [
    "Participation increased",
    "Approximately unchanged",
    "Participation decreased",
]:
    cases = freed_time_cases.loc[
        freed_time_cases["participation_status"].eq(status)
    ]

    print(f"\n{status}: {len(cases)} entities")

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
    ):
        display(
            cases.sort_values("country")[CASE_COLUMNS]
            .reset_index(drop=True)
            .round(3)
        )


Participation increased: 18 entities


,country,year_start,year_end,interval_years,female_unpaid_hours_start,female_unpaid_hours_end,freed_minutes_per_day,female_lfpr_pct_start,female_lfpr_pct_end,female_lfpr_change_pp,survey_comparability
0,Bulgaria,2002,2010,8,5.017,4.433,35.000,46.197,47.819,1.622,Not yet assessed
1,Estonia,2000,2010,10,4.783,4.133,39.000,52.575,55.069,2.494,Not yet assessed
2,Germany,2002,2013,11,4.242,3.936,18.360,49.580,54.194,4.614,Not yet assessed
3,"Hong Kong SAR, China",2002,2013,11,3.300,2.600,42.000,51.449,53.869,2.420,Not yet assessed
4,Hungary,2000,2010,10,4.683,4.146,32.215,45.552,46.638,1.086,Not yet assessed
5,Italy,2003,2014,11,5.256,4.896,21.600,36.803,39.746,2.943,Not yet assessed
6,Mexico,2002,2019,17,6.987,6.664,19.356,38.314,45.573,7.259,Not yet assessed
7,Mongolia,2000,2019,19,4.814,4.630,11.045,53.834,57.902,4.068,Not yet assessed
8,Norway,2001,2011,10,3.933,3.674,15.560,61.699,63.764,2.065,Not yet assessed
9,Oman,2000,2008,8,4.933,4.533,24.000,23.917,25.695,1.778,Not yet assessed



Approximately unchanged: 5 entities


,country,year_start,year_end,interval_years,female_unpaid_hours_start,female_unpaid_hours_end,freed_minutes_per_day,female_lfpr_pct_start,female_lfpr_pct_end,female_lfpr_change_pp,survey_comparability
0,Armenia,2004,2008,4,5.767,5.198,34.120,53.518,52.960,-0.558,Not yet assessed
1,El Salvador,2010,2017,7,5.460,4.851,36.507,45.966,45.154,-0.812,Not yet assessed
2,North Macedonia,2004,2015,11,5.700,3.700,120.000,42.490,42.307,-0.183,Not yet assessed
3,Romania,2000,2012,12,5.000,4.567,26.000,42.361,41.763,-0.598,Not yet assessed
4,Sweden,2001,2011,10,4.236,3.850,23.170,58.578,59.526,0.948,Not yet assessed



Participation decreased: 4 entities


,country,year_start,year_end,interval_years,female_unpaid_hours_start,female_unpaid_hours_end,freed_minutes_per_day,female_lfpr_pct_start,female_lfpr_pct_end,female_lfpr_change_pp,survey_comparability
0,Kazakhstan,2000,2018,18,6.650,4.550,126.0,65.834,64.818,-1.016,Not yet assessed
1,Kyrgyz Republic,2000,2015,15,6.650,4.033,157.0,57.886,47.459,-10.427,Not yet assessed
2,Thailand,2009,2015,6,3.067,2.840,13.6,64.659,61.068,-3.591,Not yet assessed
3,United States,2003,2019,16,3.970,3.680,17.4,58.224,56.868,-1.356,Not yet assessed


In [12]:
TIME_THRESHOLDS_MINUTES = [0, 5, 10, 15, 30]
PARTICIPATION_THRESHOLDS_PP = [0, 0.5, 1.0]
NUMERIC_TOLERANCE = 1e-6

sensitivity_views = {
    "First-to-last: exclude conflicts": first_last.loc[
        ~first_last["either_endpoint_flagged"]
    ].copy(),
    "Adjacent: exclude conflicts": adjacent.loc[
        ~adjacent["either_endpoint_flagged"]
    ].copy(),
}

threshold_rows = []

for view_name, frame in sensitivity_views.items():
    for minutes in TIME_THRESHOLDS_MINUTES:
        time_cutoff_hours = max(
            minutes / 60,
            NUMERIC_TOLERANCE,
        )

        selected = frame.loc[
            frame["freed_hours_per_day"].gt(time_cutoff_hours)
        ].copy()

        for threshold in PARTICIPATION_THRESHOLDS_PP:
            participation_cutoff = max(threshold, NUMERIC_TOLERANCE)
            participation_change = selected["female_lfpr_change_pp"]

            increased = participation_change.gt(participation_cutoff)
            decreased = participation_change.lt(-participation_cutoff)
            approximately_unchanged = ~(increased | decreased)

            without_increase = int(
                (approximately_unchanged | decreased).sum()
            )

            threshold_rows.append({
                "view": view_name,
                "time_reduction_over_minutes": minutes,
                "participation_threshold_pp": threshold,
                "pairs": len(selected),
                "entities": selected["iso3"].nunique(),
                "participation_increased": int(increased.sum()),
                "participation_approximately_unchanged": int(
                    approximately_unchanged.sum()
                ),
                "participation_decreased": int(decreased.sum()),
                "without_increase_under_rule": without_increase,
                "without_increase_pct": (
                    without_increase / len(selected) * 100
                    if len(selected) else np.nan
                ),
            })

threshold_sensitivity = pd.DataFrame(threshold_rows)

for view_name in sensitivity_views:
    results = threshold_sensitivity.loc[
        threshold_sensitivity["view"].eq(view_name)
    ]

    print(f"\nPercentage without an increase above threshold: {view_name}")

    display(
        results.pivot(
            index="time_reduction_over_minutes",
            columns="participation_threshold_pp",
            values="without_increase_pct",
        ).round(1)
    )

    print("Denominators for each time-reduction threshold")

    display(
        results[
            ["time_reduction_over_minutes", "pairs", "entities"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

print("\nFull results")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(threshold_sensitivity.round(2))


Percentage without an increase above threshold: First-to-last: exclude conflicts


participation_threshold_pp,0.0,0.5,1.0
time_reduction_over_minutes,,,
0,30.3,30.3,33.3
5,32.3,32.3,35.5
10,29.6,29.6,33.3
15,28.0,28.0,32.0
30,38.5,38.5,38.5


Denominators for each time-reduction threshold


,time_reduction_over_minutes,pairs,entities
0,0,33,33
1,5,31,31
2,10,27,27
3,15,25,25
4,30,13,13



Percentage without an increase above threshold: Adjacent: exclude conflicts


participation_threshold_pp,0.0,0.5,1.0
time_reduction_over_minutes,,,
0,34.5,52.7,56.4
5,33.3,45.2,47.6
10,31.4,40.0,42.9
15,31.2,40.6,43.8
30,42.1,47.4,47.4


Denominators for each time-reduction threshold


,time_reduction_over_minutes,pairs,entities
0,0,55,34
1,5,42,34
2,10,35,29
3,15,32,27
4,30,19,16



Full results


,view,time_reduction_over_minutes,participation_threshold_pp,pairs,entities,participation_increased,participation_approximately_unchanged,participation_decreased,without_increase_under_rule,without_increase_pct
0,First-to-last: exclude conflicts,0,0.0,33,33,23,0,10,10,30.30
1,First-to-last: exclude conflicts,0,0.5,33,33,23,1,9,10,30.30
2,First-to-last: exclude conflicts,0,1.0,33,33,22,6,5,11,33.33
3,First-to-last: exclude conflicts,5,0.0,31,31,21,0,10,10,32.26
4,First-to-last: exclude conflicts,5,0.5,31,31,21,1,9,10,32.26
5,First-to-last: exclude conflicts,5,1.0,31,31,20,6,5,11,35.48
6,First-to-last: exclude conflicts,10,0.0,27,27,19,0,8,8,29.63
7,First-to-last: exclude conflicts,10,0.5,27,27,19,1,7,8,29.63
8,First-to-last: exclude conflicts,10,1.0,27,27,18,5,4,9,33.33
9,First-to-last: exclude conflicts,15,0.0,25,25,18,0,7,7,28.00


In [13]:
RESULTS_DIR = (
    CHANGES_DIR.parent
    / "freed_time_opportunity"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "freed_time_summary": freed_time_summary,
    "threshold_sensitivity": threshold_sensitivity,
    "first_last_with_freed_time": first_last,
    "adjacent_with_freed_time": adjacent,
    "case_review_10min_1pp": freed_time_cases,
}

for name, table in tables.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)

print(f"Saved {len(tables)} result tables.")
print(f"Output directory: {RESULTS_DIR}")

Saved 5 result tables.
Output directory: D:\胡溪筼\Documents\2026-7Fall\Carolina Data Challenge 2026\Graduate_Dataset\analysis_results\freed_time_opportunity
